In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [2]:

corpus = [
    "deep learning is very powerful",
    "deep learning is very fun",
    "machine learning is very interesting",
    "artificial intelligence is the future, Nice",
    "i love machine learning, yeah",
    "lstm is used for sequence prediction, we use it",
    "neural networks learn patterns, good thing",
    "data science is very exciting",
    "python is very easy to learn",
    "models improve with more and more data"
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)


Number of sentences: 10

Sample corpus:
- deep learning is very powerful
- deep learning is very fun
- machine learning is very interesting
- artificial intelligence is the future, Nice
- i love machine learning, yeah
- lstm is used for sequence prediction, we use it
- neural networks learn patterns, good thing
- data science is very exciting
- python is very easy to learn
- models improve with more and more data


In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word_index = tokenizer.word_index
total_words = len(word_index)+1

print("Vocabulary Size: ", total_words)
print("\nWord Index: ")
print(word_index)

Vocabulary Size:  42

Word Index: 
{'is': 1, 'very': 2, 'learning': 3, 'deep': 4, 'machine': 5, 'learn': 6, 'data': 7, 'more': 8, 'powerful': 9, 'fun': 10, 'interesting': 11, 'artificial': 12, 'intelligence': 13, 'the': 14, 'future': 15, 'nice': 16, 'i': 17, 'love': 18, 'yeah': 19, 'lstm': 20, 'used': 21, 'for': 22, 'sequence': 23, 'prediction': 24, 'we': 25, 'use': 26, 'it': 27, 'neural': 28, 'networks': 29, 'patterns': 30, 'good': 31, 'thing': 32, 'science': 33, 'exciting': 34, 'python': 35, 'easy': 36, 'to': 37, 'models': 38, 'improve': 39, 'with': 40, 'and': 41}


In [4]:
input_sequences = []
for line in corpus:
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1, len(token_list)):
    n_gram_seq = token_list[: i+1]
    input_sequences.append(n_gram_seq)

print("Total training sequences: ", len(input_sequences))
print("\nSome sequences before padding: ")
for seq in input_sequences[:10]:
  print(seq)


Total training sequences:  49

Some sequences before padding: 
[4, 3]
[4, 3, 1]
[4, 3, 1, 2]
[4, 3, 1, 2, 9]
[4, 3]
[4, 3, 1]
[4, 3, 1, 2]
[4, 3, 1, 2, 10]
[5, 3]
[5, 3, 1]


In [5]:
max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen = max_seq_len, padding = 'pre')

print("Maximum seq length: ", max_seq_len)
print("\n Padded sequence: ")
print(input_sequences[:10])

Maximum seq length:  9

 Padded sequence: 
[[ 0  0  0  0  0  0  0  4  3]
 [ 0  0  0  0  0  0  4  3  1]
 [ 0  0  0  0  0  4  3  1  2]
 [ 0  0  0  0  4  3  1  2  9]
 [ 0  0  0  0  0  0  0  4  3]
 [ 0  0  0  0  0  0  4  3  1]
 [ 0  0  0  0  0  4  3  1  2]
 [ 0  0  0  0  4  3  1  2 10]
 [ 0  0  0  0  0  0  0  5  3]
 [ 0  0  0  0  0  0  5  3  1]]


In [6]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (49, 8)
Shape of y: (49, 42)


In [7]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:

history = model.fit(X, y, epochs=200, verbose=1)


Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.0204 - loss: 3.7389
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1429 - loss: 3.7336
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1429 - loss: 3.7293
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1429 - loss: 3.7240
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1429 - loss: 3.7174
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1429 - loss: 3.7097
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1429 - loss: 3.6997
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1429 - loss: 3.6856
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1429 - loss: 3.6670
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1429 - loss: 3.6359
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1429 - loss: 3.5960
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1429 - lo

In [9]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text


In [10]:

print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))


deep learning is very
machine learning is very
python is very easy


In [11]:

# Try your own inputs here
print(generate_text("artificial intelligence", 2))
print(generate_text("data science", 2))
print(generate_text("lstm is", 3))


artificial intelligence is the
data science is very
lstm is used for sequence


MYCODE

In [12]:
# importing libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [13]:
# CREATING CORPUS
corpus = [
    "Domain-Specific Relevance",
    "Improved Model Accuracy",
    "Capturing Specific Vocabulary",
    "Better Handling of Language Variations",
    "Importing Necessary Libraries",
    "Loading the Data",
    "Preprocessing the Text",
    "Saving the Processed Data",
    "model making and training",
    "structured collection of text",
    "Understanding Custom Corpus",
    "specialized collection of text data",
    "creating a custom corpus",
    "achieve better performance",
    "NLP model understands and processes"
]
print("Number of scentences: ", len(corpus))
print("\sample corpus: ")
for line in corpus:
  print("-", line)

Number of scentences:  15
\sample corpus: 
- Domain-Specific Relevance
- Improved Model Accuracy
- Capturing Specific Vocabulary
- Better Handling of Language Variations
- Importing Necessary Libraries
- Loading the Data
- Preprocessing the Text
- Saving the Processed Data
- model making and training
- structured collection of text
- Understanding Custom Corpus
- specialized collection of text data
- creating a custom corpus
- achieve better performance
- NLP model understands and processes


<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1766/1573743418.py:20: SyntaxWarning: invalid escape sequence '\s'
  print("\sample corpus: ")


In [14]:
# TOKENIZATION
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

index_word = tokenizer.word_index
total_words = len(index_word)+1

print("Vocabulary size: ", total_words)
print("\nWord Index: ")
print(index_word)

Vocabulary size:  40

Word Index: 
{'model': 1, 'of': 2, 'the': 3, 'data': 4, 'text': 5, 'specific': 6, 'better': 7, 'and': 8, 'collection': 9, 'custom': 10, 'corpus': 11, 'domain': 12, 'relevance': 13, 'improved': 14, 'accuracy': 15, 'capturing': 16, 'vocabulary': 17, 'handling': 18, 'language': 19, 'variations': 20, 'importing': 21, 'necessary': 22, 'libraries': 23, 'loading': 24, 'preprocessing': 25, 'saving': 26, 'processed': 27, 'making': 28, 'training': 29, 'structured': 30, 'understanding': 31, 'specialized': 32, 'creating': 33, 'a': 34, 'achieve': 35, 'performance': 36, 'nlp': 37, 'understands': 38, 'processes': 39}


In [15]:
input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)

Total training sequences: 40

Some sequences before padding:
[12, 6]
[12, 6, 13]
[14, 1]
[14, 1, 15]
[16, 6]
[16, 6, 17]
[7, 18]
[7, 18, 2]
[7, 18, 2, 19]
[7, 18, 2, 19, 20]


In [16]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])


Maximum sequence length: 5

Padded sequences:
[[ 0  0  0 12  6]
 [ 0  0 12  6 13]
 [ 0  0  0 14  1]
 [ 0  0 14  1 15]
 [ 0  0  0 16  6]
 [ 0  0 16  6 17]
 [ 0  0  0  7 18]
 [ 0  0  7 18  2]
 [ 0  7 18  2 19]
 [ 7 18  2 19 20]]


In [17]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (40, 4)
Shape of y: (40, 40)


In [18]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:

history = model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.0000e+00 - loss: 3.6890
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0750 - loss: 3.6844
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1000 - loss: 3.6804
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1000 - loss: 3.6759
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1000 - loss: 3.6716
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1000 - loss: 3.6662
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1000 - loss: 3.6607 
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1250 - loss: 3.6541
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1250 - loss: 3.6462
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1250 - loss: 3.6374
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0750 - loss: 3.6266
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0750

In [20]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text


In [21]:

print(generate_text("Improved Model", 1))
print(generate_text("Loading the", 1))
print(generate_text("Saving the", 2))


Improved Model accuracy
Loading the data
Saving the processed data
